In [1]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()  # Load environment variables from .env file
openai_client = OpenAI()  # Initialize the OpenAI client

In [2]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [3]:
from rag_helper import RAGBase

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
)

In [5]:
# Importing our sentence transformer model
from sentence_transformers import SentenceTransformer

# Creating a model object
model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Text search performs well but we can still avoid many issues if we use vector search.

In [4]:
# we add the embedder to our RAG class then we use it to vectorize to the query and search the index for the most relevant documents inside the search method.
class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs): # stores an embedder instance (plus any RAGBase kwargs)
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {"course": self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

 The embedder is passed as an argument to the RAGVector class and is used to encode the query into a vector representation. The search method then uses this vector to perform a similarity search in the index, returning the most relevant documents based on the query.

In [ ]:
# Creating an instance of RAGVector with the embedder, index, and llm_client
vector_assistant = RAGVector(
    embedder = model,
    index=index,
    llm_client=openai_client,
)

In [7]:
query = "I just found the program, can I sign up?"
vector_results = vector_assistant.rag(query)

AttributeError: 'numpy.ndarray' object has no attribute 'lower'